## Data Import

In [1]:
import pandas as pd

# Read Excel file with all sheets
excel_file = 'data/AMMO_IM_AI-V01.xlsx'

# Read each sheet into a separate DataFrame
df_inventory = pd.read_excel(excel_file, sheet_name='Inventory')
df_hmsd = pd.read_excel(excel_file, sheet_name='HMSD')
df_sloc = pd.read_excel(excel_file, sheet_name='SLOC')
df_hd = pd.read_excel(excel_file, sheet_name='HD')
df_cg = pd.read_excel(excel_file, sheet_name='CG')

print(f"Inventory shape: {df_inventory.shape}")
print(f"HMSD shape: {df_hmsd.shape}")
print(f"SLOC shape: {df_sloc.shape}")
print(f"HD shape: {df_hd.shape}")
print(f"CG shape: {df_cg.shape}")

Inventory shape: (70, 9)
HMSD shape: (48, 7)
SLOC shape: (5, 2)
HD shape: (11, 2)
CG shape: (13, 2)


In [2]:
df_inventory.head()

,Material,Description,Hazard Division,CG,NEW (EA - KG),Storage Location,SLOC Desc,Quantity,NEW (Total - KG)
0,885600034,POINT DETONATING FUZE,1.2,B,0.0218,199A,Location A,57,1.2426
1,888093467,"CTG, 5.56MM 4 AP M995/1 TR M856",1.4,S,0.0021,199A,Location A,12,0.0252
2,886182049,"CTG, 7.62MM AP M993 SNGL RD",1.4,C,0.0029,199A,Location A,83,0.2407
3,884708253,"CTG, CAL .50 4 API M8/1 API-DT MK257-0",1.4,G,0.0164,199A,Location A,29,0.4756
4,880083529,"CTG, 105MM CLEARING (1315)",1.3,C,1.3154,199A,Location A,46,60.5084


In [3]:
df_hmsd.head()

,Internchangability Code,FSC,Material,Material Description,HD,CG,NEW (KG)
0,0364,1390,885600034,POINT DETONATING FUZE,1.2,B,0.0218
1,0381,1310,883317492,"CARTRIDGES, POWER DEVICE",1.2,C,0.0546
2,AA02,1305,888093467,"CTG, 5.56MM 4 AP M995/1 TR M856",1.4,S,0.0021
3,AA03,1305,886182049,"CTG, 7.62MM AP M993 SNGL RD",1.4,C,0.0029
4,AA05,1305,887612509,"CTG, CAL .50 1 API MK211-0/1 AP M2/1 API-T M20",1.2.2,G,0.0006


In [4]:
df_sloc.head()

,SLOC,SLOC Description
0,199A,Location A
1,199B,Location B
2,199C,Location C
3,199D,Location D
4,199E,Location E


In [5]:
df_hd.head()

,HD,HD DESCRIPTION
0,1.1,Ammunition that has a mass explosion hazard.
1,1.2,Ammunition that has a projection hazard not of...
2,1.2.1,More hazardous items of HD 1.2 large fragments...
3,1.2.2,More hazardous items of HD 1.2 small fragments...
4,1.2.3,Explosion reaction during sympathetic reaction...


In [6]:
df_cg.head()

,CG,CG DESCRIPTION
0,A,Primary explosive substance.
1,B,Articles containing a primary explosive substance
2,C,Propellant explosive substance
3,D,Secondary detonating article
4,E,secondary detonating exp w/o its own means of ...


---

## HANA persistence

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
from hdbcli import dbapi

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

# Connect to HANA
conn = dbapi.connect(
    address=address,
    port=port,
    user=user,
    password=password
)

cursor = conn.cursor()
print("Connected to HANA successfully!")

# Drop schema if exists to recreate from scratch
try:
    cursor.execute('DROP SCHEMA "EXPLOSIVES" CASCADE')
    print("Dropped existing EXPLOSIVES schema")
except Exception as e:
    print(f"Schema drop note: {e}")

# Create EXPLOSIVES schema
try:
    cursor.execute('CREATE SCHEMA "EXPLOSIVES"')
    print("Schema EXPLOSIVES created successfully!")
except Exception as e:
    print(f"Schema creation note: {e}")
    # Schema might already exist, which is okays

Connected to HANA successfully!
Dropped existing EXPLOSIVES schema
Schema EXPLOSIVES created successfully!


### Create tables

In [8]:
# Create INVENTORY table
create_inventory_sql = '''
CREATE COLUMN TABLE "EXPLOSIVES"."INVENTORY" (
    "Material" INTEGER,
    "Description" NVARCHAR(500),
    "Hazard_Division" NVARCHAR(50),
    "CG" NVARCHAR(50),
    "NEW_EA_KG" DOUBLE,
    "Storage_Location" NVARCHAR(50),
    "SLOC_Desc" NVARCHAR(500),
    "Quantity" INTEGER,
    "NEW_Total_KG" DOUBLE,
    PRIMARY KEY ("Material", "Storage_Location")
)
'''

try:
    cursor.execute(create_inventory_sql)
    print("INVENTORY table created successfully!")
except Exception as e:
    print(f"INVENTORY table: {e}")

# Create HMSD table
create_hmsd_sql = '''
CREATE COLUMN TABLE "EXPLOSIVES"."HMSD" (
    "Interchangeability_Code" NVARCHAR(100),
    "FSC" INTEGER,
    "Material" INTEGER,
    "Material_Description" NVARCHAR(500),
    "HD" NVARCHAR(50),
    "CG" NVARCHAR(50),
    "NEW_KG" DOUBLE,
    PRIMARY KEY ("Material")
)
'''

try:
    cursor.execute(create_hmsd_sql)
    print("HMSD table created successfully!")
except Exception as e:
    print(f"HMSD table: {e}")

# Create SLOC table
create_sloc_sql = '''
CREATE COLUMN TABLE "EXPLOSIVES"."SLOC" (
    "SLOC" NVARCHAR(50),
    "SLOC_Description" NVARCHAR(500),
    PRIMARY KEY ("SLOC")
)
'''

try:
    cursor.execute(create_sloc_sql)
    print("SLOC table created successfully!")
except Exception as e:
    print(f"SLOC table: {e}")

# Create HD table
create_hd_sql = '''
CREATE COLUMN TABLE "EXPLOSIVES"."HD" (
    "HD" NVARCHAR(50),
    "HD_Description" NVARCHAR(500),
    PRIMARY KEY ("HD")
)
'''

try:
    cursor.execute(create_hd_sql)
    print("HD table created successfully!")
except Exception as e:
    print(f"HD table: {e}")

# Create CG table
create_cg_sql = '''
CREATE COLUMN TABLE "EXPLOSIVES"."CG" (
    "CG" NVARCHAR(50),
    "CG_Description" NVARCHAR(500),
    PRIMARY KEY ("CG")
)
'''

try:
    cursor.execute(create_cg_sql)
    print("CG table created successfully!")
except Exception as e:
    print(f"CG table: {e}")

INVENTORY table created successfully!
HMSD table created successfully!
SLOC table created successfully!
HD table created successfully!
CG table created successfully!


In [9]:
# Prepare and insert INVENTORY data
df_inventory_clean = df_inventory.copy()
df_inventory_clean.columns = ['Material', 'Description', 'Hazard_Division', 'CG', 'NEW_EA_KG', 
                                'Storage_Location', 'SLOC_Desc', 'Quantity', 'NEW_Total_KG']

insert_inventory_sql = '''
INSERT INTO "EXPLOSIVES"."INVENTORY" 
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
'''

inserted_count = 0
for _, row in df_inventory_clean.iterrows():
    try:
        cursor.execute(insert_inventory_sql, tuple(row))
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting inventory row: {e}")

conn.commit()
print(f"Inserted {inserted_count} rows into INVENTORY table")

# Prepare and insert HMSD data
df_hmsd_clean = df_hmsd.copy()
df_hmsd_clean.columns = ['Interchangeability_Code', 'FSC', 'Material', 'Material_Description', 
                          'HD', 'CG', 'NEW_KG']

insert_hmsd_sql = '''
INSERT INTO "EXPLOSIVES"."HMSD" 
VALUES (?, ?, ?, ?, ?, ?, ?)
'''

inserted_count = 0
for _, row in df_hmsd_clean.iterrows():
    try:
        cursor.execute(insert_hmsd_sql, tuple(row))
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting HMSD row: {e}")

conn.commit()
print(f"Inserted {inserted_count} rows into HMSD table")

# Prepare and insert SLOC data
df_sloc_clean = df_sloc.copy()
df_sloc_clean.columns = ['SLOC', 'SLOC_Description']

insert_sloc_sql = '''
INSERT INTO "EXPLOSIVES"."SLOC" 
VALUES (?, ?)
'''

inserted_count = 0
for _, row in df_sloc_clean.iterrows():
    try:
        cursor.execute(insert_sloc_sql, tuple(row))
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting SLOC row: {e}")

conn.commit()
print(f"Inserted {inserted_count} rows into SLOC table")

# Prepare and insert HD data
df_hd_clean = df_hd.copy()
df_hd_clean.columns = ['HD', 'HD_Description']

insert_hd_sql = '''
INSERT INTO "EXPLOSIVES"."HD" 
VALUES (?, ?)
'''

inserted_count = 0
for _, row in df_hd_clean.iterrows():
    try:
        cursor.execute(insert_hd_sql, tuple(row))
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting HD row: {e}")

conn.commit()
print(f"Inserted {inserted_count} rows into HD table")

# Prepare and insert CG data
df_cg_clean = df_cg.copy()
df_cg_clean.columns = ['CG', 'CG_Description']

insert_cg_sql = '''
INSERT INTO "EXPLOSIVES"."CG" 
VALUES (?, ?)
'''

inserted_count = 0
for _, row in df_cg_clean.iterrows():
    try:
        cursor.execute(insert_cg_sql, tuple(row))
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting CG row: {e}")

conn.commit()
print(f"Inserted {inserted_count} rows into CG table")

Inserted 70 rows into INVENTORY table
Inserted 48 rows into HMSD table
Inserted 5 rows into SLOC table
Inserted 11 rows into HD table
Inserted 13 rows into CG table


In [10]:
# Verify the data was inserted correctly
print("=== Verification Summary ===\n")

cursor.execute('SELECT COUNT(*) FROM "EXPLOSIVES"."INVENTORY"')
print(f"INVENTORY: {cursor.fetchone()[0]} rows")

cursor.execute('SELECT COUNT(*) FROM "EXPLOSIVES"."HMSD"')
print(f"HMSD: {cursor.fetchone()[0]} rows")

cursor.execute('SELECT COUNT(*) FROM "EXPLOSIVES"."SLOC"')
print(f"SLOC: {cursor.fetchone()[0]} rows")

cursor.execute('SELECT COUNT(*) FROM "EXPLOSIVES"."HD"')
print(f"HD: {cursor.fetchone()[0]} rows")

cursor.execute('SELECT COUNT(*) FROM "EXPLOSIVES"."CG"')
print(f"CG: {cursor.fetchone()[0]} rows")

print("\nAll tables created and populated successfully!")

# Close the connection
cursor.close()
conn.close()

=== Verification Summary ===

INVENTORY: 70 rows
HMSD: 48 rows
SLOC: 5 rows
HD: 11 rows
CG: 13 rows

All tables created and populated successfully!


----

## S/4 Data Pull

In [37]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import os
from dotenv import load_dotenv
from hdbcli import dbapi
load_dotenv()

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

# Connect to HANA
conn = dbapi.connect(
    address=address,
    port=port,
    user=user,
    password=password
)

cursor = conn.cursor()

sql = "SELECT * FROM EXPLOSIVES.INVENTORY"
cursor.execute(sql)

result = cursor.fetchall() # Get result
df_inv = pd.DataFrame(result, columns=[desc[0] for desc in cursor.description])

sql = "SELECT * FROM EXPLOSIVES.HMSD"
cursor.execute(sql)

result = cursor.fetchall() # Get result
df_hmsd = pd.DataFrame(result, columns=[desc[0] for desc in cursor.description])

sql = "SELECT * FROM EXPLOSIVES.SLOC"
cursor.execute(sql)

result = cursor.fetchall() # Get result
df_sloc = pd.DataFrame(result, columns=[desc[0] for desc in cursor.description])

# Close the connection
cursor.close()
conn.close()

# HANA Credentials
s4_user = os.getenv("S4_USER")
s4_password = os.getenv("S4_PASSWORD")

sloc = ["00AJ", "00AK", "00AL", "00AM", "00AN", "00AO", "00AP", "00AQ", "00AR"]

df_inventory = pd.DataFrame()

for sloc_code in sloc:
    url = f"https://coeportal515.saphosting.de/sap/opu/odata/sap/API_MATERIAL_STOCK_SRV/A_MatlStkInAcctMod?$format=json&sap-client=600&$filter=StorageLocation%20eq%20%27{sloc_code}%27"
    response = requests.get(url, auth=HTTPBasicAuth(s4_user, s4_password))
    
    if response.status_code == 200:
        # Convert JSON response to a dictionary
        data = response.json()
    
        # Extract the data you need (in this case, the entity data)
        entity_data = data.get('d', {}).get('results', [])
    
        # Load data into a Pandas DataFrame
        df_inventory_x = pd.DataFrame(entity_data)
    else:
        print(f"Failed to fetch data: {response.status_code}")

    df_inventory = pd.concat([df_inventory, df_inventory_x], ignore_index=True)

# Filter the relevant columns
df_inventory = df_inventory[['Material', 'StorageLocation', 'Batch', 'MaterialBaseUnit', 'MatlWrhsStkQtyInMatlBaseUnit']]

# Join with HMSD data
df_inventory["Material"] = df_inventory["Material"].apply(lambda x: int(x))
df_inventory["MatlWrhsStkQtyInMatlBaseUnit"] = df_inventory["MatlWrhsStkQtyInMatlBaseUnit"].apply(lambda x: int(x))
df_inventory = df_inventory.merge(df_hmsd[["Material", "Material_Description", "HD", "CG", "NEW_KG"]], on="Material", how="left")

# Join with SLOC data
df_inventory = df_inventory.merge(df_sloc[["SLOC", "SLOC_Description"]], left_on="StorageLocation", right_on="SLOC", how="left")

# Pick out and rename relevant columns post-join
df_inventory = df_inventory[["Material", "Material_Description", "HD", "CG", "NEW_KG", "StorageLocation", "SLOC_Description", "Batch", "MaterialBaseUnit", "MatlWrhsStkQtyInMatlBaseUnit"]]

# Calculate new total KG
df_inventory["NEW_Total_KG"] = df_inventory["NEW_KG"] * df_inventory["MatlWrhsStkQtyInMatlBaseUnit"]

# Rename columns
df_inventory.columns = [
    "Material", 
    "Description", 
    "Hazard_Division", 
    "CG", 
    "NEW_EA_KG", 
    "Storage_Location", 
    "SLOC_Desc", 
    "Batch", 
    "MaterialBaseUnit", 
    "Quantity", 
    "NEW_Total_KG"
]

df_inventory.head()

,Material,Description,Hazard_Division,CG,NEW_EA_KG,Storage_Location,SLOC_Desc,Batch,MaterialBaseUnit,Quantity,NEW_Total_KG
0,885600034,POINT DETONATING FUZE,1.2,B,0.0218,00AJ,SLOC 00AJ EXP_AI,AKJDQMKPC,EA,57,1.2426
1,888093467,"CTG, 5.56MM 4 AP M995/1 TR M856",1.4,S,0.0021,00AJ,SLOC 00AJ EXP_AI,AFB4SBCKU,EA,12,0.0252
2,886182049,"CTG, 7.62MM AP M993 SNGL RD",1.4,C,0.0029,00AJ,SLOC 00AJ EXP_AI,AZ3TMDE9K,EA,83,0.2407
3,884708253,"CTG, CAL .50 4 API M8/1 API-DT MK257-0",1.4,G,0.0164,00AJ,SLOC 00AJ EXP_AI,ARXUAXASE,EA,29,0.4756
4,880083529,"CTG, 105MM CLEARING (1315)",1.3,C,1.3154,00AJ,SLOC 00AJ EXP_AI,APMJL5WCM,EA,46,60.5084


In [38]:
df_inv.head()

,Material,Description,Hazard_Division,CG,NEW_EA_KG,Storage_Location,SLOC_Desc,Quantity,NEW_Total_KG
0,885600034,POINT DETONATING FUZE,1.2,B,0.0218,00AJ,SLOC 00AJ EXP_AI,57,1.2426
1,888093467,"CTG, 5.56MM 4 AP M995/1 TR M856",1.4,S,0.0021,00AJ,SLOC 00AJ EXP_AI,12,0.0252
2,886182049,"CTG, 7.62MM AP M993 SNGL RD",1.4,C,0.0029,00AJ,SLOC 00AJ EXP_AI,83,0.2407
3,884708253,"CTG, CAL .50 4 API M8/1 API-DT MK257-0",1.4,G,0.0164,00AJ,SLOC 00AJ EXP_AI,29,0.4756
4,880083529,"CTG, 105MM CLEARING (1315)",1.3,C,1.3154,00AJ,SLOC 00AJ EXP_AI,46,60.5084


---

## AI Analysis

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from hdbcli import dbapi
import pandas as pd

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

# AI Core Credentials
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

from gen_ai_hub.proxy.native.openai import chat

In [3]:
def base_prompt(text):
    
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"You are an explosives expert asked to provide insights on different types of explosives and their effects, supported by data extracted from an underlying database." +
                            f"Here is the original user prompt: {text}." +
                            # This is where we will add the context queried from the different tables!
                            f"Reply to the original user prompt to the best of your knowledge and ability."
                }
            ]
        }
    ]
    return messages

def write_prompt_messages(messages, model_name = "gpt-5"): # Models can be changed here
    """Send messages to the model and return the response."""
    kwargs = dict(model_name=model_name, messages=messages)
    response = chat.completions.create(**kwargs)
    return response.to_dict()["choices"][0]["message"]["content"]

def get_prompt_answer(text):
    """Get response from GPT 5"""
    messages = base_prompt(text)
    answer = write_prompt_messages(messages)
    return answer

get_prompt_answer("Tell me about C4 explosives and their effects.")

'I can share high-level, non-actionable information about C4 and its effects. I will not provide guidance on making, acquiring, or using explosives.\n\nOverview\n- C4 is a category of plastic explosive whose energetic component is primarily RDX, blended with binders and plasticizers to form a tough, putty-like material.\n- It is valued in professional demolition and military applications for being stable, relatively insensitive to accidental shock and heat, and easy to shape to a target. It still requires a proper detonator to initiate; flame or small impacts alone typically won’t detonate it.\n\nRepresentative properties and performance (typical ranges)\n- Physical form: malleable, rubbery solid; can be cut and pressed to surfaces.\n- Density: roughly 1.5–1.6 g/cm³.\n- Energetic strength: greater brisance and shattering power than TNT on a per-mass basis; TNT equivalence commonly cited around 1.3.\n- Detonation velocity: on the order of 8 km/s (varies with formulation, density, and co

### Compatibility Matrix

![Compatibility Matrix](./data/compatibility_matrix.png)

In [4]:
import numpy as np

# Explosive Compatibility Groups
compatibility_groups = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'N', 'S']

compatibility_matrix = np.array([
    ["X", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0"],  # A
    ["0", "X", "1", "1", "1", "1", "1", "0", "0", "0", "0", "0", "X"],  # B
    ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # C
    ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # D
    ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # E
    ["0", "1", "2", "2", "2", "X", "2,3", "0", "0", "0", "0", "0", "X"],  # F
    ["1", "3", "3", "3", "2,3", "X", "0", "0", "0", "0", "0", "0", "X"],  # G
    ["0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0", "0", "X"],  # H
    ["0", "0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0", "X"],  # J
    ["0", "0", "0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0"],  # K
    ["0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "5", "0", "0"],  # L
    ["0", "0", "4", "4", "4", "0", "0", "0", "0", "0", "0", "6", "7"],  # N
    ["0", "X", "X", "X", "X", "X", "X", "X", "X", "0", "0", "7", "X"],  # S
])

# Print compatibility information
print("Legend Descriptions:")
print("-" * 50)
descriptions = {
    '0': 'Not permitted',
    'X': 'Permitted',
    '1': 'Permitted with warning: Compatibility Group B fuzes may be stored with the articles to which they will be assembled, but the Net Explosive Quantity (NEQ) shall be aggregated and treated as Compatibility Group F.',
    '2': 'Permitted with warning: Storage in the same building may be permitted if effectively segregated to prevent propagation.',
    '3': 'Permitted with warning: Mixing of articles of Compatibility Group G with articles of other compatibility groups is at the discretion of the National Competent Authority.',
    '2,3': 'Permitted with warning: Storage in the same building may be permitted if effectively segregated to prevent propagation. Mixing of articles of Compatibility Group G with articles of other compatibility groups is at the discretion of the National Competent Authority.',
    '4': 'Permitted with warning: Articles of Compatibility Group N should not in general be stored with articles in other compatibility groups except S. However, if such articles are stored with articles of Compatibility Group C, D and E, the articles of Compatibility Group N should be considered as having the characteristics of Compatibility Group D and the compatibility groups mixing rules apply accordingly.',
    '5': 'Permitted with warning: Compatibility Group L articles shall always be stored separately from all articles of other compatibility groups as well as from all other articles of different types of Compatibility Group L.',
    '6': 'Permitted with warning: It is allowed to mix 1.6N munitions. The Compatibility Group of the mixed set remains N if the munitions belong to the same family or if it has been demonstrated that, in case of a detonation of one munition, there is no instant transmission to the munitions of another family (the families are then called ‘compatible’). If it is not the case the whole set of munitions should be considered as having the characteristics of Compatibility Group D.',
    '7': 'A mixed set of munitions 1.6N and 1.4S may be considered as having the characteristics of Compatibility Group N.'
}

for legend, desc in descriptions.items():
    print(f"Legend {legend}: {desc}")

Legend Descriptions:
--------------------------------------------------
Legend 0: Not permitted
Legend X: Permitted
Legend 1: Permitted with warning: Compatibility Group B fuzes may be stored with the articles to which they will be assembled, but the Net Explosive Quantity (NEQ) shall be aggregated and treated as Compatibility Group F.
Legend 2: Permitted with warning: Storage in the same building may be permitted if effectively segregated to prevent propagation.
Legend 3: Permitted with warning: Mixing of articles of Compatibility Group G with articles of other compatibility groups is at the discretion of the National Competent Authority.
Legend 2,3: Permitted with warning: Storage in the same building may be permitted if effectively segregated to prevent propagation. Mixing of articles of Compatibility Group G with articles of other compatibility groups is at the discretion of the National Competent Authority.
Legend 4: Permitted with warning: Articles of Compatibility Group N should

---
### Material-based functionality

In [5]:
import numpy as np

def get_compatibility_group(matnr): # Helper function to find the compatibility group of a material

    # Connect to HANA
    conn = dbapi.connect(
        address=address,
        port=port,
        user=user,
        password=password
    )

    cursor = conn.cursor()

    cursor.execute(f'''SELECT "EXPLOSIVES"."INVENTORY"."CG", "EXPLOSIVES"."CG"."CG_Description",
                   "EXPLOSIVES"."INVENTORY"."Description", "EXPLOSIVES"."INVENTORY"."Hazard_Division"
                   FROM "EXPLOSIVES"."INVENTORY" JOIN "EXPLOSIVES"."CG" ON "EXPLOSIVES"."INVENTORY"."CG" = "EXPLOSIVES"."CG"."CG" 
                   WHERE "EXPLOSIVES"."INVENTORY"."Material" = {matnr} LIMIT 1''')
    result = cursor.fetchone()

    # Close the connection
    cursor.close()
    conn.close()

    return result

def get_compatibility_info(material_number): # Main function to get compatibility info
    result_row = get_compatibility_group(material_number)
    result_group = result_row[0]
    result_group_desc = result_row[1]
    result_material_desc = result_row[2]
    result_hazard_division = result_row[3]
    if not result_row:
        return None

    compatibility_groups = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'N', 'S']

    compatibility_matrix = np.array([
        ["X", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0"],  # A
        ["0", "X", "1", "1", "1", "1", "1", "0", "0", "0", "0", "0", "X"],  # B
        ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # C
        ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # D
        ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # E
        ["0", "1", "2", "2", "2", "X", "2,3", "0", "0", "0", "0", "0", "X"],  # F
        ["1", "3", "3", "3", "2,3", "X", "0", "0", "0", "0", "0", "0", "X"],  # G
        ["0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0", "0", "X"],  # H
        ["0", "0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0", "X"],  # J
        ["0", "0", "0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0"],  # K
        ["0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "5", "0", "0"],  # L
        ["0", "0", "4", "4", "4", "0", "0", "0", "0", "0", "0", "6", "7"],  # N
        ["0", "X", "X", "X", "X", "X", "X", "X", "X", "0", "0", "7", "X"],  # S
    ])

    # Get the row for the material's compatibility group
    comp_index = compatibility_groups.index(result_row[0])
    compatibility_row = compatibility_matrix[comp_index]

    # Pair each compatibility group with its compatibility value (convert to regular strings)
    paired_dict = {cg: str(compatibility_row[i]) for i, cg in enumerate(compatibility_groups)}
    
    return result_group, paired_dict, result_group_desc, result_material_desc, result_hazard_division


result_group, result_info, result_group_desc, result_material_desc, result_hazard_division = get_compatibility_info(885600034)
print("Result group:", result_group)
print("Result compatibility info:", result_info)
print("Result group description:", result_group_desc)
print("Result material description:", result_material_desc)
print("Result hazard division:", result_hazard_division)

Result group: B
Result compatibility info: {'A': '0', 'B': 'X', 'C': '1', 'D': '1', 'E': '1', 'F': '1', 'G': '1', 'H': '0', 'J': '0', 'K': '0', 'L': '0', 'N': '0', 'S': 'X'}
Result group description: Articles containing a primary explosive substance
Result material description: POINT DETONATING FUZE
Result hazard division: 1.2


In [6]:
def base_prompt(matnr):

    result_group, result_info, result_group_desc, result_material_desc, result_hazard_division = get_compatibility_info(matnr)

    # Compatibility Groups Descriptions
    cgs_dict = {
        'A': 'Primary explosive substance.',
        'B': 'Articles containing a primary explosive substance',
        'C': 'Propellant explosive substance',
        'D': 'Secondary detonating article',
        'E': 'secondary detonating exp w/o its own means of initiation',
        'F': 'secondary detonating exp with its own means of initiation',
        'G': 'Pyrotechnic substance',
        'H': 'Article containing both explosive substance and white ph',
        'J': 'Ammunition containing both explosives and flammable liquids',
        'K': 'Articles containing both an explosive substance and a toxic',
        'L': 'Explosive substance w special risk needing isolation',
        'N': 'Hazard Division 1.6 ammunition containing only EIDS',
        'S': 'Substance w haz effects frm accidental functioning in package'
    }

    # Compatibility Matrix Descriptions
    descriptions = {
        '0': 'Not permitted',
        'X': 'Permitted',
        '1': 'Permitted with warning: Compatibility Group B fuzes may be stored with the articles to which they will be assembled, but the Net Explosive Quantity (NEQ) shall be aggregated and treated as Compatibility Group F.',
        '2': 'Permitted with warning: Storage in the same building may be permitted if effectively segregated to prevent propagation.',
        '3': 'Permitted with warning: Mixing of articles of Compatibility Group G with articles of other compatibility groups is at the discretion of the National Competent Authority.',
        '2,3': 'Permitted with warning: Storage in the same building may be permitted if effectively segregated to prevent propagation. Mixing of articles of Compatibility Group G with articles of other compatibility groups is at the discretion of the National Competent Authority.',
        '4': 'Permitted with warning: Articles of Compatibility Group N should not in general be stored with articles in other compatibility groups except S. However, if such articles are stored with articles of Compatibility Group C, D and E, the articles of Compatibility Group N should be considered as having the characteristics of Compatibility Group D and the compatibility groups mixing rules apply accordingly.',
        '5': 'Permitted with warning: Compatibility Group L articles shall always be stored separately from all articles of other compatibility groups as well as from all other articles of different types of Compatibility Group L.',
        '6': 'Permitted with warning: It is allowed to mix 1.6N munitions. The Compatibility Group of the mixed set remains N if the munitions belong to the same family or if it has been demonstrated that, in case of a detonation of one munition, there is no instant transmission to the munitions of another family (the families are then called ‘compatible’). If it is not the case the whole set of munitions should be considered as having the characteristics of Compatibility Group D.',
        '7': 'A mixed set of munitions 1.6N and 1.4S may be considered as having the characteristics of Compatibility Group N.'
    }
    
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"You are an explosives expert asked to provide insights on different types of explosives and their effects, supported by data extracted from an underlying database." +
                            f"Here are the different possible compatibility groups and their descriptions: {cgs_dict}." +
                            f"Here are the compatibility matrix descriptions: {descriptions}." +
                            f"You have been tasked with identifying the compatibility information for a specific explosive material." +
                            f"The material number is {matnr}, which corresponds to the material description '{result_material_desc}' and falls under Hazard Division '{result_hazard_division}'." +
                            f"The material belongs to Compatibility Group '{result_group}', described as '{result_group_desc}'." +
                            f"The compatibility information for this group with respect to other groups is as follows: {result_info}." +
                            f"Using this information, provide a detailed analysis of the compatibility of this material with other explosive materials, highlighting any special considerations or warnings that should be observed."
                }
            ]
        }
    ]
    return messages

def write_prompt_messages(messages, model_name = "gpt-5"): # Models can be changed here
    """Send messages to the model and return the response."""
    kwargs = dict(model_name=model_name, messages=messages)
    response = chat.completions.create(**kwargs)
    return response.to_dict()["choices"][0]["message"]["content"]

def get_prompt_answer(text):
    """Get response from GPT 5"""
    messages = base_prompt(text)
    answer = write_prompt_messages(messages)
    return answer

In [6]:
fuze_answer = get_prompt_answer(885600034)
print(fuze_answer)

Material overview
- Material number: 885600034
- Description: POINT DETONATING FUZE
- Hazard Division: 1.2 (projection hazard; no mass explosion)
- Compatibility Group: B (Articles containing a primary explosive substance)

General safety context for Compatibility Group B
- Group B articles contain small quantities of primary explosives integrated into an article (e.g., fuzes). Primary explosives are highly sensitive to heat, shock, and friction. Even though HD 1.2 indicates no mass explosion hazard, the sensitivity of Group B means mixing with many other groups is restricted or only allowed with specific controls.
- A special rule applies when Group B fuzes are stored with the articles to which they will be assembled: the Net Explosive Quantity (NEQ) must be aggregated and treated as Compatibility Group F for storage and quantity–distance purposes.

Compatibility by group (based on the provided matrix)
- Group A (Primary explosive substance): Not permitted (code 0)
  - Rationale: Comb

### Next Step: SQL Generation

In [ ]:
def generate_sql_from_prompt(user_prompt):
    """
    LLM agent that generates SQL statements for HANA explosives database.
    
    Args:
        user_prompt (str): Natural language query from the user
        
    Returns:
        str: Generated SQL statement
    """
    
    # Database schema description
    schema_context = """
You are an expert SQL developer working with a SAP HANA database named EXPLOSIVES.

The database contains 5 tables with the following schema:

1. Table: EXPLOSIVES.CG (Compatibility Groups)
   - CG (NVARCHAR(50), PRIMARY KEY): Compatibility Group code (e.g., 'A', 'B', 'C', etc.)
   - CG_Description (NVARCHAR(500)): Description of the compatibility group
   
2. Table: EXPLOSIVES.HD (Hazard Divisions)
   - HD (NVARCHAR(50), PRIMARY KEY): Hazard Division code
   - HD_Description (NVARCHAR(500)): Description of the hazard division

3. Table: EXPLOSIVES.SLOC (Storage Locations)
   - SLOC (NVARCHAR(50), PRIMARY KEY): Storage Location code
   - SLOC_Description (NVARCHAR(500)): Description of the storage location

4. Table: EXPLOSIVES.HMSD (Hazardous Substance Master Data)
   - Interchangeability_Code (NVARCHAR(100)): Interchangeability code for ammunition
   - FSC (INTEGER): Federal Supply Code
   - Material (INTEGER, PRIMARY KEY): Material number (NSN material type)
   - Material_Description (NVARCHAR(500)): Description of the material
   - HD (NVARCHAR(50)): Hazard Division (references EXPLOSIVES.HD.HD)
   - CG (NVARCHAR(50)): Compatibility Group (references EXPLOSIVES.CG.CG)
   - NEW_KG (DOUBLE): Net Explosive Weight (NEW) in KG

5. Table: EXPLOSIVES.INVENTORY (Inventory Data)
   - Material (INTEGER, PRIMARY KEY composite): Material number (references EXPLOSIVES.HMSD.Material)
   - Description (NVARCHAR(500)): Material description (lookup from HMSD)
   - Hazard_Division (NVARCHAR(50)): Hazard Division (lookup from HMSD)
   - CG (NVARCHAR(50)): Compatibility Group (lookup from HMSD)
   - NEW_EA_KG (DOUBLE): NEW per unit (ea) in KG (lookup from HMSD)
   - Storage_Location (NVARCHAR(50), PRIMARY KEY composite): Storage location code
   - SLOC_Desc (NVARCHAR(500)): Storage location description (lookup from SLOC)
   - Quantity (INTEGER): Quantity in each (ea)
   - NEW_Total_KG (DOUBLE): Total NEW in KG (NEW_EA_KG * Quantity)

Key Relationships:
- INVENTORY.Material → HMSD.Material (material master data)
- INVENTORY.CG → CG.CG (compatibility group info)
- INVENTORY.Hazard_Division → HD.HD (hazard division info)
- INVENTORY.Storage_Location → SLOC.SLOC (storage location info)
- HMSD.CG → CG.CG (compatibility group info)
- HMSD.HD → HD.HD (hazard division info)

Important Notes:
- Use double quotes for schema and table names: "EXPLOSIVES"."INVENTORY"
- Use double quotes for column names: "Material", "Description"
- For SAP HANA, use LIMIT for result limiting
- NEW stands for Net Explosive Weight (also known as Net Explosive Quantity/NEQ)
- All text searches should be case-insensitive where appropriate
"""

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": schema_context
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"""Based on the database schema provided, generate a SQL query for the following request:

{user_prompt}

Requirements:
- Return ONLY the SQL statement, no explanations
- Use proper SAP HANA SQL syntax with double quotes
- Ensure the query is optimized and follows best practices
- Use appropriate JOINs when needed to get related data
- Include relevant WHERE clauses for filtering
- Use LIMIT when appropriate to avoid excessive results"""
                }
            ]
        }
    ]
    
    # Call GPT-3.5-turbo to generate SQL
    kwargs = dict(model_name="gpt-35-turbo", messages=messages)
    response = chat.completions.create(**kwargs)
    sql_statement = response.to_dict()["choices"][0]["message"]["content"]
    
    # Clean up the response (remove markdown code blocks if present)
    sql_statement = sql_statement.strip()
    if sql_statement.startswith("```sql"):
        sql_statement = sql_statement[6:]
    if sql_statement.startswith("```"):
        sql_statement = sql_statement[3:]
    if sql_statement.endswith("```"):
        sql_statement = sql_statement[:-3]
    sql_statement = sql_statement.strip()
    
    return sql_statement


# Test the SQL generation agent
test_queries = [
    "Show me all materials with compatibility group 'D' and their total NEW",
    "List all storage locations with their descriptions",
    "Find materials that have more than 100 units in inventory",
    "What are all the compatibility groups and their descriptions?"
]

print("=== SQL Generation Agent Test ===\n")
for query in test_queries:
    print(f"Query: {query}")
    print("-" * 80)
    sql = generate_sql_from_prompt(query)
    print(f"Generated SQL:\n{sql}\n")
    print("=" * 80)
    print()

In [12]:
# Test the SQL generation agent on HANA

def get_hana_from_language(prompt):

    # Connect to HANA
    conn = dbapi.connect(
        address=address,
        port=port,
        user=user,
        password=password
    )

    cursor = conn.cursor()

    sql = generate_sql_from_prompt(prompt)
    print(f"Executing SQL:\n{sql}\n")
    cursor.execute(sql)

    result = cursor.fetchall() # Get result

    # Close the connection
    cursor.close()
    conn.close()

    return result

# Test the function straightaway
resulting_rows = get_hana_from_language("Show me all materials with compatibility group 'D' and their total NEW")
resulting_df = pd.DataFrame(resulting_rows)
resulting_df.columns = ['Material', 'Description', 'CG', 'NEW_KG']

resulting_df

Executing SQL:
SELECT 
    "HMSD"."Material",
    "HMSD"."Material_Description",
    "HMSD"."CG",
    SUM("INVENTORY"."NEW_Total_KG") AS "Total_NEW_KG"
FROM 
    "EXPLOSIVES"."HMSD"
JOIN 
    "EXPLOSIVES"."INVENTORY" 
    ON "HMSD"."Material" = "INVENTORY"."Material"
WHERE 
    "HMSD"."CG" = 'D'
GROUP BY 
    "HMSD"."Material",
    "HMSD"."Material_Description",
    "HMSD"."CG"
LIMIT 100;



,Material,Description,CG,NEW_KG
0,880074186,"BURSTER, PROJ M54A1",D,8.1393
1,882316804,"BOMB, GUIDED GP GBU-39/B SDB",D,335.6580
2,888193642,"CTG ASSY, F/LT WT EARTH ANCHOR (1377)",D,0.4535


-----

## Agentic Setup

In [45]:
# ============================================================================
# EXPLOSIVES INTELLIGENCE AGENT - Standalone Implementation
# ============================================================================

import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from hdbcli import dbapi
import requests
from requests.auth import HTTPBasicAuth

# Load environment variables
load_dotenv()

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

# S/4 Credentials
s4_user = os.getenv("S4_USER")
s4_password = os.getenv("S4_PASSWORD")

# AI Core Credentials
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")

os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

from gen_ai_hub.proxy.native.openai import chat

# ============================================================================
# HELPER FUNCTION - Fetch Inventory from S/4
# ============================================================================

def fetch_inventory_from_s4():
    """
    Fetch current inventory data from S/4 HANA and join with HANA master data.
    Returns a DataFrame with complete inventory information.
    """
    try:
        # Connect to HANA for master data
        conn = dbapi.connect(address=address, port=port, user=user, password=password)
        cursor = conn.cursor()
        
        # Get HMSD data
        cursor.execute('SELECT * FROM "EXPLOSIVES"."HMSD"')
        result = cursor.fetchall()
        df_hmsd = pd.DataFrame(result, columns=[desc[0] for desc in cursor.description])
        
        # Get SLOC data
        cursor.execute('SELECT * FROM "EXPLOSIVES"."SLOC"')
        result = cursor.fetchall()
        df_sloc = pd.DataFrame(result, columns=[desc[0] for desc in cursor.description])
        
        cursor.close()
        conn.close()
        
        # Fetch from S/4
        sloc_codes = ["00AJ", "00AK", "00AL", "00AM", "00AN", "00AO", "00AP", "00AQ", "00AR"]
        df_inventory = pd.DataFrame()
        
        for sloc_code in sloc_codes:
            url = f"https://coeportal515.saphosting.de/sap/opu/odata/sap/API_MATERIAL_STOCK_SRV/A_MatlStkInAcctMod?$format=json&sap-client=600&$filter=StorageLocation%20eq%20%27{sloc_code}%27"
            response = requests.get(url, auth=HTTPBasicAuth(s4_user, s4_password), timeout=30)
            
            if response.status_code == 200:
                data = response.json()
                entity_data = data.get('d', {}).get('results', [])
                df_inventory_x = pd.DataFrame(entity_data)
                df_inventory = pd.concat([df_inventory, df_inventory_x], ignore_index=True)
        
        if df_inventory.empty:
            return pd.DataFrame()
        
        # Filter relevant columns
        df_inventory = df_inventory[['Material', 'StorageLocation', 'Batch', 'MaterialBaseUnit', 'MatlWrhsStkQtyInMatlBaseUnit']]
        
        # Convert types
        df_inventory["Material"] = df_inventory["Material"].apply(lambda x: int(x))
        df_inventory["MatlWrhsStkQtyInMatlBaseUnit"] = df_inventory["MatlWrhsStkQtyInMatlBaseUnit"].apply(lambda x: int(x))
        
        # Join with HMSD
        df_inventory = df_inventory.merge(
            df_hmsd[["Material", "Material_Description", "HD", "CG", "NEW_KG"]], 
            on="Material", 
            how="left"
        )
        
        # Join with SLOC
        df_inventory = df_inventory.merge(
            df_sloc[["SLOC", "SLOC_Description"]], 
            left_on="StorageLocation", 
            right_on="SLOC", 
            how="left"
        )
        
        # Select and rename columns
        df_inventory = df_inventory[[
            "Material", "Material_Description", "HD", "CG", "NEW_KG", 
            "StorageLocation", "SLOC_Description", "Batch", "MaterialBaseUnit", 
            "MatlWrhsStkQtyInMatlBaseUnit"
        ]]
        
        # Calculate total NEW
        df_inventory["NEW_Total_KG"] = df_inventory["NEW_KG"] * df_inventory["MatlWrhsStkQtyInMatlBaseUnit"]
        
        # Rename columns for consistency
        df_inventory.columns = [
            "Material", "Description", "Hazard_Division", "CG", "NEW_EA_KG",
            "Storage_Location", "SLOC_Desc", "Batch", "MaterialBaseUnit",
            "Quantity", "NEW_Total_KG"
        ]
        
        return df_inventory
        
    except Exception as e:
        print(f"Error fetching inventory from S/4: {str(e)}")
        return pd.DataFrame()

# ============================================================================
# IATG NEW AGGREGATION CLASS
# ============================================================================

class IATGAggregator:
    """
    IATG (International Ammunition Technical Guidelines) NEW Aggregation.
    
    Implements IATG rules for aggregating Net Explosive Weight (NEW) when multiple
    Hazard Divisions (HD) are stored together in the same location. This is used
    for calculating Quantity Distance (QTD) requirements.
    """
    
    def __init__(self, inventory):
        """
        Args:
            inventory: dict of HD: weight (e.g., {"1.2": 50, "1.3.1": 100})
        """
        self.raw_inv = inventory
        # Standardizing all variants into their primary IATG aggregation buckets
        self.weights = {
            "1.1": inventory.get("1.1", 0) + inventory.get("1.5", 0),
            # IATG Rule: HD 1.2 (unspecified) and 1.2.3 are treated as 1.2.1
            "1.2.1": (inventory.get("1.2.1", 0) + 
                      inventory.get("1.2.3", 0) + 
                      inventory.get("1.2", 0)),
            "1.2.2": inventory.get("1.2.2", 0),
            "1.3": (inventory.get("1.3", 0) + 
                    inventory.get("1.3.1", 0) + 
                    inventory.get("1.3.2", 0)),
            "1.4": inventory.get("1.4", 0),
            "1.6": inventory.get("1.6", 0)
        }

    def calculate_aggregation(self, verbose=False):
        """
        Calculate the aggregated HD and NEW according to IATG rules.
        
        Args:
            verbose: If True, print step-by-step trace
            
        Returns:
            tuple: (aggregated_hd, aggregated_new_kg)
        """
        if verbose:
            print("--- IATG Step-by-Step Aggregation Trace ---")
            for hd, weight in self.raw_inv.items():
                if weight > 0:
                    print(f"Input: {hd} = {weight} kg")
            print("-" * 40)

        # 1. The 1.1/1.5 Rule - Aggregates everything EXCEPT 1.4
        if self.weights["1.1"] > 0:
            total = sum(self.weights.values()) - self.weights["1.4"]
            if verbose:
                print(f"[RULE 1] HD 1.1/1.5 detected.")
                print(f"         Aggregating ALL items to 1.1 except 1.4 which is ignored: {total} kg")
                if self.weights["1.4"] > 0:
                    print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.1", total

        # 2. The 1.2.1 Rule (Includes general 1.2 and 1.2.3)
        if self.weights["1.2.1"] > 0:
            total = (self.weights["1.2.1"] + self.weights["1.2.2"] + 
                     self.weights["1.3"] + self.weights["1.6"])
            if verbose:
                print(f"[RULE 2] HD 1.2.1 (or generic 1.2) detected.")
                print(f"         Aggregating 1.2.x, 1.3, and 1.6 as 1.2.1: {total} kg")
                if self.weights["1.4"] > 0:
                    print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")            
            return "1.2.1", total

        # 3. The 1.2.2 Rule
        if self.weights["1.2.2"] > 0:
            total = self.weights["1.2.2"] + self.weights["1.3"] + self.weights["1.6"]
            if verbose:
                print(f"[RULE 3] HD 1.2.2 detected.")
                print(f"         Aggregating 1.2.2, 1.3, and 1.6 as 1.2.2: {total} kg")
                if self.weights["1.4"] > 0:
                    print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.2.2", total

        # 4. The 1.3 Rule
        if self.weights["1.3"] > 0:
            total = self.weights["1.3"] + self.weights["1.6"]
            if verbose:
                print(f"[RULE 4] HD 1.3 detected.")
                print(f"         Aggregating 1.3 and 1.6 as 1.3: {total} kg")
                if self.weights["1.4"] > 0:
                    print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.3", total

        # 5. The 1.6 Rule
        if self.weights["1.6"] > 0:
            if verbose:
                print(f"[RULE 5] Only HD 1.6 and 1.4 present. 1.4 is ignored")
                if self.weights["1.4"] > 0:
                    print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.6", self.weights["1.6"]

        # 6. The 1.4 Rule
        if verbose:
            print(f"[RULE 6] Only HD 1.4")
        return "1.4", self.weights["1.4"]

# ============================================================================
# TOOL FUNCTIONS - These will be available to the LLM
# ============================================================================

def get_compatibility_info_tool(material_number):
    """
    Get comprehensive compatibility information for a specific explosive material.
    Returns compatibility group, descriptions, and compatibility matrix.
    """
    try:
        # Connect to HANA for master data
        conn = dbapi.connect(address=address, port=port, user=user, password=password)
        cursor = conn.cursor()
        
        # Query material compatibility group and details from HMSD
        cursor.execute(f'''
            SELECT "EXPLOSIVES"."HMSD"."CG", 
                   "EXPLOSIVES"."CG"."CG_Description",
                   "EXPLOSIVES"."HMSD"."Material_Description", 
                   "EXPLOSIVES"."HMSD"."HD"
            FROM "EXPLOSIVES"."HMSD" 
            JOIN "EXPLOSIVES"."CG" ON "EXPLOSIVES"."HMSD"."CG" = "EXPLOSIVES"."CG"."CG" 
            WHERE "EXPLOSIVES"."HMSD"."Material" = {material_number} 
            LIMIT 1
        ''')
        result = cursor.fetchone()
        
        cursor.close()
        conn.close()
        
        if not result:
            return {"error": f"Material {material_number} not found in database"}
        
        result_group = result[0]
        result_group_desc = result[1]
        result_material_desc = result[2]
        result_hazard_division = result[3]
        
        # Compatibility matrix
        compatibility_groups = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'N', 'S']
        compatibility_matrix = np.array([
            ["X", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0"],  # A
            ["0", "X", "1", "1", "1", "1", "1", "0", "0", "0", "0", "0", "X"],  # B
            ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # C
            ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # D
            ["0", "1", "X", "X", "X", "2", "3", "0", "0", "0", "0", "4", "X"],  # E
            ["0", "1", "2", "2", "2", "X", "2,3", "0", "0", "0", "0", "0", "X"],  # F
            ["1", "3", "3", "3", "2,3", "X", "0", "0", "0", "0", "0", "0", "X"],  # G
            ["0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0", "0", "X"],  # H
            ["0", "0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0", "X"],  # J
            ["0", "0", "0", "0", "0", "0", "0", "0", "0", "X", "0", "0", "0"],  # K
            ["0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "5", "0", "0"],  # L
            ["0", "0", "4", "4", "4", "0", "0", "0", "0", "0", "0", "6", "7"],  # N
            ["0", "X", "X", "X", "X", "X", "X", "X", "X", "0", "0", "7", "X"],  # S
        ])
        
        comp_index = compatibility_groups.index(result_group)
        compatibility_row = compatibility_matrix[comp_index]
        paired_dict = {cg: str(compatibility_row[i]) for i, cg in enumerate(compatibility_groups)}
        
        return {
            "material_number": material_number,
            "material_description": result_material_desc,
            "compatibility_group": result_group,
            "compatibility_group_description": result_group_desc,
            "hazard_division": result_hazard_division,
            "compatibility_with_other_groups": paired_dict,
            "compatibility_legend": {
                "0": "Not permitted",
                "X": "Permitted",
                "1": "Permitted with warning: CG B fuzes may be stored with articles they'll be assembled with",
                "2": "Permitted with warning: Storage in same building if effectively segregated",
                "3": "Permitted with warning: Mixing CG G is at discretion of National Competent Authority",
                "2,3": "Permitted with warnings for both segregation and CG G mixing",
                "4": "Permitted with warning: CG N stored with C/D/E should be considered as CG D",
                "5": "Permitted with warning: CG L must always be stored separately",
                "6": "Permitted with warning: 1.6N munitions mixing rules apply",
                "7": "Permitted with warning: Mixed 1.6N and 1.4S may be considered as CG N"
            }
        }
    except Exception as e:
        return {"error": f"Error retrieving compatibility info: {str(e)}"}


def query_database_with_sql_tool(user_query):
    """
    Generate SQL from natural language query and execute it against the HANA database.
    Note: Inventory data is fetched from S/4, not from HANA tables.
    Returns query results as structured data.
    """
    try:
        # Generate SQL using LLM
        schema_context = """
You are an expert SQL developer working with a SAP HANA database named EXPLOSIVES.

The database contains 4 tables with the following schema:

1. Table: EXPLOSIVES.CG (Compatibility Groups)
   - CG (NVARCHAR(50), PRIMARY KEY): Compatibility Group code
   - CG_Description (NVARCHAR(500)): Description

2. Table: EXPLOSIVES.HD (Hazard Divisions)
   - HD (NVARCHAR(50), PRIMARY KEY): Hazard Division code
   - HD_Description (NVARCHAR(500)): Description

3. Table: EXPLOSIVES.SLOC (Storage Locations)
   - SLOC (NVARCHAR(50), PRIMARY KEY): Storage Location code
   - SLOC_Description (NVARCHAR(500)): Description

4. Table: EXPLOSIVES.HMSD (Hazardous Substance Master Data)
   - Material (INTEGER, PRIMARY KEY): Material number
   - Material_Description (NVARCHAR(500)): Description
   - HD (NVARCHAR(50)): Hazard Division
   - CG (NVARCHAR(50)): Compatibility Group
   - NEW_KG (DOUBLE): Net Explosive Weight in KG
   - FSC (INTEGER): Federal Supply Code
   - Interchangeability_Code (NVARCHAR(100))

IMPORTANT NOTES:
- The INVENTORY table is NO LONGER AVAILABLE in HANA. Inventory data is now fetched from S/4 HANA via API.
- For inventory-related queries, you can only query the HMSD table for material master data.
- To get actual inventory quantities and storage locations, use the get_material_inventory tool instead.
- Use double quotes for schema/table/column names.
- Use LIMIT to restrict results.
"""
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": schema_context}]},
            {"role": "user", "content": [{"type": "text", "text": f"Generate SQL for: {user_query}\nReturn ONLY the SQL statement. If this requires inventory data (quantities, storage locations, etc.), inform the user to use the inventory tool instead."}]}
        ]
        
        response = chat.completions.create(model_name="gpt-35-turbo", messages=messages)
        sql_statement = response.to_dict()["choices"][0]["message"]["content"].strip()
        
        # Clean SQL
        if sql_statement.startswith("```sql"):
            sql_statement = sql_statement[6:]
        if sql_statement.startswith("```"):
            sql_statement = sql_statement[3:]
        if sql_statement.endswith("```"):
            sql_statement = sql_statement[:-3]
        sql_statement = sql_statement.strip()
        
        # Execute SQL
        conn = dbapi.connect(address=address, port=port, user=user, password=password)
        cursor = conn.cursor()
        cursor.execute(sql_statement)
        
        # Fetch results
        results = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]
        
        cursor.close()
        conn.close()
        
        # Format results as list of dictionaries
        formatted_results = []
        for row in results[:50]:  # Limit to 50 rows for readability
            formatted_results.append(dict(zip(column_names, row)))
        
        return {
            "query": user_query,
            "sql_generated": sql_statement,
            "row_count": len(results),
            "results": formatted_results,
            "note": "Results limited to 50 rows" if len(results) > 50 else "All results shown"
        }
        
    except Exception as e:
        return {"error": f"Error executing database query: {str(e)}"}


def get_material_inventory_tool(material_number):
    """
    Get detailed inventory information for a specific material across all storage locations.
    Data is fetched live from S/4 HANA system.
    """
    try:
        # Fetch current inventory from S/4
        df_inventory = fetch_inventory_from_s4()
        
        if df_inventory.empty:
            return {"error": "Unable to fetch inventory data from S/4"}
        
        # Filter for specific material
        material_inventory = df_inventory[df_inventory["Material"] == material_number]
        
        if material_inventory.empty:
            return {"error": f"No inventory found for material {material_number}"}
        
        inventory_list = []
        total_quantity = 0
        total_new = 0
        
        for _, row in material_inventory.iterrows():
            inventory_list.append({
                "material": int(row["Material"]),
                "description": row["Description"],
                "hazard_division": row["Hazard_Division"],
                "compatibility_group": row["CG"],
                "new_per_unit_kg": float(row["NEW_EA_KG"]) if pd.notna(row["NEW_EA_KG"]) else 0,
                "storage_location": row["Storage_Location"],
                "storage_location_description": row["SLOC_Desc"],
                "batch": row["Batch"],
                "base_unit": row["MaterialBaseUnit"],
                "quantity": int(row["Quantity"]),
                "total_new_kg": float(row["NEW_Total_KG"]) if pd.notna(row["NEW_Total_KG"]) else 0
            })
            total_quantity += int(row["Quantity"])
            total_new += float(row["NEW_Total_KG"]) if pd.notna(row["NEW_Total_KG"]) else 0
        
        return {
            "material_number": material_number,
            "inventory_locations": inventory_list,
            "total_quantity_all_locations": total_quantity,
            "total_new_all_locations_kg": total_new,
            "number_of_storage_locations": len(inventory_list),
            "data_source": "S/4 HANA (live data)"
        }
        
    except Exception as e:
        return {"error": f"Error retrieving inventory: {str(e)}"}


def list_all_materials_tool():
    """
    List all explosive materials in the database with basic information.
    """
    try:
        conn = dbapi.connect(address=address, port=port, user=user, password=password)
        cursor = conn.cursor()
        
        cursor.execute('''
            SELECT DISTINCT "Material", "Material_Description", "HD", "CG", "NEW_KG"
            FROM "EXPLOSIVES"."HMSD"
            ORDER BY "Material"
            LIMIT 100
        ''')
        
        results = cursor.fetchall()
        cursor.close()
        conn.close()
        
        materials_list = []
        for row in results:
            materials_list.append({
                "material_number": row[0],
                "description": row[1],
                "hazard_division": row[2],
                "compatibility_group": row[3],
                "new_kg": row[4]
            })
        
        return {
            "total_materials": len(materials_list),
            "materials": materials_list,
            "note": "Limited to 100 materials for readability. Use get_material_inventory to check current stock levels from S/4."
        }
        
    except Exception as e:
        return {"error": f"Error listing materials: {str(e)}"}


def get_storage_summary_tool():
    """
    Get a summary of all storage locations and their total explosive quantities.
    Data is fetched live from S/4 HANA system.
    Includes IATG-aggregated NEW values for Quantity Distance calculations.
    """
    try:
        # Fetch current inventory from S/4
        df_inventory = fetch_inventory_from_s4()
        
        if df_inventory.empty:
            return {"error": "Unable to fetch inventory data from S/4"}
        
        # Group by storage location
        storage_summary = df_inventory.groupby(['Storage_Location', 'SLOC_Desc']).agg({
            'Material': 'nunique',
            'Quantity': 'sum',
            'NEW_Total_KG': 'sum'
        }).reset_index()
        
        storage_summary.columns = ['Storage_Location', 'SLOC_Desc', 'Material_Count', 'Total_Items', 'Total_NEW_KG']
        storage_summary = storage_summary.sort_values('Total_NEW_KG', ascending=False)
        
        storage_list = []
        for _, row in storage_summary.iterrows():
            location = row["Storage_Location"]
            
            # Calculate IATG aggregated NEW for this location
            location_inventory = df_inventory[df_inventory["Storage_Location"] == location]
            hd_weights = {}
            for _, inv_row in location_inventory.iterrows():
                hd = inv_row["Hazard_Division"]
                new_kg = float(inv_row["NEW_Total_KG"]) if pd.notna(inv_row["NEW_Total_KG"]) else 0
                if pd.notna(hd) and hd:
                    hd_weights[hd] = hd_weights.get(hd, 0) + new_kg
            
            # Apply IATG aggregation
            aggregated_hd = None
            aggregated_new = 0
            if hd_weights:
                aggregator = IATGAggregator(hd_weights)
                aggregated_hd, aggregated_new = aggregator.calculate_aggregation(verbose=False)
            
            storage_list.append({
                "storage_location": row["Storage_Location"],
                "description": row["SLOC_Desc"],
                "unique_materials": int(row["Material_Count"]),
                "total_items": int(row["Total_Items"]),
                "total_new_kg": float(row["Total_NEW_KG"]) if pd.notna(row["Total_NEW_KG"]) else 0,
                "iatg_aggregated_hd": aggregated_hd,
                "iatg_aggregated_new_kg": round(aggregated_new, 4) if aggregated_new else 0,
                "hazard_divisions_present": list(hd_weights.keys()) if hd_weights else []
            })
        
        return {
            "total_storage_locations": len(storage_list),
            "storage_locations": storage_list,
            "data_source": "S/4 HANA (live data)",
            "note": "IATG aggregated values are provided for Quantity Distance (QTD) calculations"
        }
        
    except Exception as e:
        return {"error": f"Error retrieving storage summary: {str(e)}"}


def calculate_storage_location_qtd_tool(storage_location):
    """
    Calculate IATG-aggregated Hazard Division and NEW for a specific storage location.
    This is used for Quantity Distance (QTD) calculations and safety planning.
    """
    try:
        # Fetch current inventory from S/4
        df_inventory = fetch_inventory_from_s4()
        
        if df_inventory.empty:
            return {"error": "Unable to fetch inventory data from S/4"}
        
        # Filter for specific storage location
        location_inventory = df_inventory[df_inventory["Storage_Location"] == storage_location]
        
        if location_inventory.empty:
            return {"error": f"No inventory found for storage location {storage_location}"}
        
        # Get storage location description
        sloc_desc = location_inventory.iloc[0]["SLOC_Desc"] if not location_inventory.empty else "Unknown"
        
        # Group by Hazard Division and sum NEW
        hd_weights = {}
        materials_by_hd = {}
        
        for _, row in location_inventory.iterrows():
            hd = row["Hazard_Division"]
            new_kg = float(row["NEW_Total_KG"]) if pd.notna(row["NEW_Total_KG"]) else 0
            material = int(row["Material"])
            
            if pd.notna(hd) and hd:
                hd_weights[hd] = hd_weights.get(hd, 0) + new_kg
                if hd not in materials_by_hd:
                    materials_by_hd[hd] = []
                materials_by_hd[hd].append({
                    "material": material,
                    "description": row["Description"],
                    "new_kg": new_kg
                })
        
        if not hd_weights:
            return {"error": f"No hazard division data found for storage location {storage_location}"}
        
        # Apply IATG aggregation
        aggregator = IATGAggregator(hd_weights)
        aggregated_hd, aggregated_new = aggregator.calculate_aggregation(verbose=False)
        
        # Prepare detailed breakdown
        hd_breakdown = []
        for hd, weight in hd_weights.items():
            hd_breakdown.append({
                "hazard_division": hd,
                "total_new_kg": round(weight, 4),
                "material_count": len(materials_by_hd.get(hd, [])),
                "materials": materials_by_hd.get(hd, [])
            })
        
        return {
            "storage_location": storage_location,
            "storage_location_description": sloc_desc,
            "hazard_division_breakdown": hd_breakdown,
            "iatg_aggregated_hd": aggregated_hd,
            "iatg_aggregated_new_kg": round(aggregated_new, 4),
            "total_materials": int(location_inventory["Material"].nunique()),
            "total_items": int(location_inventory["Quantity"].sum()),
            "iatg_rule_applied": f"Aggregated to HD {aggregated_hd} per IATG guidelines",
            "data_source": "S/4 HANA (live data)",
            "usage": "Use aggregated values for Quantity Distance (QTD) and safety distance calculations"
        }
        
    except Exception as e:
        return {"error": f"Error calculating QTD for storage location: {str(e)}"}


def calculate_materials_aggregation_tool(material_numbers):
    """
    Calculate IATG-aggregated Hazard Division and NEW for a list of specific materials.
    This "what-if" tool helps determine the aggregated HD and NEW if specific materials
    were stored together, useful for planning and safety assessment.
    """
    try:
        # Connect to HANA for material master data
        conn = dbapi.connect(address=address, port=port, user=user, password=password)
        cursor = conn.cursor()
        
        # Build material list for query
        material_list = ",".join(str(m) for m in material_numbers)
        
        # Get material data including HD and NEW
        cursor.execute(f'''
            SELECT "Material", "Material_Description", "HD", "NEW_KG"
            FROM "EXPLOSIVES"."HMSD"
            WHERE "Material" IN ({material_list})
        ''')
        
        results = cursor.fetchall()
        cursor.close()
        conn.close()
        
        if not results:
            return {"error": f"None of the specified materials found in database"}
        
        # Build HD weights and material details
        hd_weights = {}
        materials_by_hd = {}
        materials_info = []
        
        for row in results:
            material = row[0]
            description = row[1]
            hd = row[2]
            new_kg = float(row[3]) if row[3] is not None else 0
            
            materials_info.append({
                "material": material,
                "description": description,
                "hazard_division": hd,
                "new_per_unit_kg": new_kg
            })
            
            if hd:
                hd_weights[hd] = hd_weights.get(hd, 0) + new_kg
                if hd not in materials_by_hd:
                    materials_by_hd[hd] = []
                materials_by_hd[hd].append({
                    "material": material,
                    "description": description,
                    "new_kg": new_kg
                })
        
        if not hd_weights:
            return {"error": "No valid hazard division data found for specified materials"}
        
        # Apply IATG aggregation
        aggregator = IATGAggregator(hd_weights)
        aggregated_hd, aggregated_new = aggregator.calculate_aggregation(verbose=False)
        
        # Prepare HD breakdown
        hd_breakdown = []
        for hd, weight in hd_weights.items():
            hd_breakdown.append({
                "hazard_division": hd,
                "total_new_kg": round(weight, 4),
                "material_count": len(materials_by_hd.get(hd, [])),
                "materials": materials_by_hd.get(hd, [])
            })
        
        # Check for missing materials
        found_materials = {row[0] for row in results}
        missing_materials = [m for m in material_numbers if m not in found_materials]
        
        return {
            "materials_analyzed": materials_info,
            "material_count": len(materials_info),
            "hazard_division_breakdown": hd_breakdown,
            "iatg_aggregated_hd": aggregated_hd,
            "iatg_aggregated_new_kg": round(aggregated_new, 4),
            "iatg_rule_applied": f"If stored together, these materials would be aggregated to HD {aggregated_hd} per IATG guidelines",
            "missing_materials": missing_materials if missing_materials else None,
            "usage": "Use aggregated values for Quantity Distance (QTD) calculations if these materials are stored together"
        }
        
    except Exception as e:
        return {"error": f"Error calculating materials aggregation: {str(e)}"}


# ============================================================================
# TOOL DEFINITIONS FOR LLM
# ============================================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_compatibility_info",
            "description": "Get comprehensive compatibility information for a specific explosive material, including its compatibility group, hazard division, and compatibility with other explosive groups. Use this when user asks about a specific material's compatibility or storage requirements.",
            "parameters": {
                "type": "object",
                "properties": {
                    "material_number": {
                        "type": "integer",
                        "description": "The material number (e.g., 885600034)"
                    }
                },
                "required": ["material_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_database_with_sql",
            "description": "Execute a natural language query against the explosives master data in HANA. Note: This only accesses master data tables (HMSD, CG, HD, SLOC). For current inventory quantities and storage locations, use get_material_inventory or get_storage_summary instead. Use this for queries about material properties, compatibility groups, hazard divisions, or storage location descriptions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_query": {
                        "type": "string",
                        "description": "Natural language description of what master data to retrieve from the database"
                    }
                },
                "required": ["user_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_material_inventory",
            "description": "Get detailed LIVE inventory information for a specific material across all storage locations from S/4 HANA, including current quantities, batches, and total NET Explosive Weight (NEW). Use this when user asks about inventory levels, stock, or where a specific material is currently stored.",
            "parameters": {
                "type": "object",
                "properties": {
                    "material_number": {
                        "type": "integer",
                        "description": "The material number to check inventory for"
                    }
                },
                "required": ["material_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_all_materials",
            "description": "List all explosive materials in the master data with their basic properties (material number, description, hazard division, compatibility group, NEW per unit). Note: This shows what materials exist but NOT current inventory quantities. Use get_material_inventory to check actual stock levels.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_storage_summary",
            "description": "Get a summary of all storage locations with current total quantities, NEW values, and IATG-aggregated NEW for QTD calculations from LIVE S/4 HANA data. Use this when user asks about storage capacity, utilization, or wants an overview of where explosives are currently stored. Includes aggregated Hazard Division per IATG rules.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_storage_location_qtd",
            "description": "Calculate the IATG-aggregated Hazard Division and NEW for a specific storage location. This applies International Ammunition Technical Guidelines (IATG) rules to determine the aggregated HD and NEW used for Quantity Distance (QTD) and safety distance calculations. Use this when user asks about QTD, safety distances, or what HD to use for a specific storage location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "storage_location": {
                        "type": "string",
                        "description": "The storage location code (e.g., '00AJ', '00AK')"
                    }
                },
                "required": ["storage_location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_materials_aggregation",
            "description": "Calculate the IATG-aggregated Hazard Division and NEW for a specific list of materials. This 'what-if' analysis tool determines what the aggregated HD and NEW would be if the specified materials were stored together. Use this when user asks about combining specific materials, planning storage, or wants to know the aggregated NEW/HD for multiple materials. Example: 'What if I store materials 885600034 and 885600011 together?'",
            "parameters": {
                "type": "object",
                "properties": {
                    "material_numbers": {
                        "type": "array",
                        "items": {
                            "type": "integer"
                        },
                        "description": "List of material numbers to aggregate (e.g., [885600034, 885600011])"
                    }
                },
                "required": ["material_numbers"]
            }
        }
    }
]

# Function dispatcher
available_functions = {
    "get_compatibility_info": get_compatibility_info_tool,
    "query_database_with_sql": query_database_with_sql_tool,
    "get_material_inventory": get_material_inventory_tool,
    "list_all_materials": list_all_materials_tool,
    "get_storage_summary": get_storage_summary_tool,
    "calculate_storage_location_qtd": calculate_storage_location_qtd_tool,
    "calculate_materials_aggregation": calculate_materials_aggregation_tool
}

# ============================================================================
# AGENT EXECUTION FUNCTION
# ============================================================================

def run_explosives_agent(user_question, max_iterations=10, model="gpt-5", verbose=True):
    """
    Run the explosives intelligence agent to answer user questions.
    
    Args:
        user_question (str): The user's question about explosives
        max_iterations (int): Maximum number of tool calls allowed
        model (str): Model to use (gpt-4o, gpt-35-turbo, etc.)
        verbose (bool): Print tool calls and intermediate steps
    
    Returns:
        str: The agent's final answer
    """
    
    system_message = """You are an expert explosives safety and inventory management assistant. 
You have access to a comprehensive database of explosive materials, their compatibility groups, 
hazard divisions, and LIVE inventory data from S/4 HANA.

Your role is to:
- Answer questions about explosive materials and their properties
- Provide compatibility information and storage guidance
- Query LIVE inventory levels and storage locations from S/4 HANA
- Calculate IATG-aggregated NEW values for Quantity Distance (QTD) calculations
- Explain safety considerations and compatibility rules
- Generate insights from the explosives database

IMPORTANT: 
- Inventory data (quantities, storage locations, batches) is fetched LIVE from S/4 HANA
- For QTD and safety distance calculations, use the calculate_storage_location_qtd tool
- IATG aggregation rules are applied automatically for mixed hazard divisions
- Always use get_material_inventory or get_storage_summary for current stock information

Always provide clear, accurate, and safety-focused information. When discussing compatibility,
reference the specific compatibility codes (0, X, 1-7) and explain their meanings."""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_question}
    ]
    
    if verbose:
        print("💥 EXPLOSIVES INTELLIGENCE AGENT")
        print("=" * 80)
        print(f"🔍 Question: {user_question}\n")
    
    for iteration in range(max_iterations):
        # Call the model
        response = chat.completions.create(
            model_name=model,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        response_dict = response.to_dict()
        response_message = response_dict["choices"][0]["message"]
        
        # Check if the model wants to call a function
        tool_calls = response_message.get("tool_calls")
        
        if not tool_calls:
            # No more function calls, we have the final answer
            final_answer = response_message.get("content")
            if verbose:
                print(f"\n✅ Final Answer:\n{final_answer}")
                print("\n" + "=" * 80)
            return final_answer
        
        # Add assistant's response to messages
        messages.append(response_message)
        
        # Execute each function call
        for tool_call in tool_calls:
            function_name = tool_call["function"]["name"]
            function_args = json.loads(tool_call["function"]["arguments"])
            
            if verbose:
                print(f"🔧 Tool Call: {function_name}")
                print(f"   Parameters: {function_args}")
            
            # Call the function
            try:
                function_response = available_functions[function_name](**function_args)
                if verbose:
                    print(f"   ✓ Response received ({len(json.dumps(function_response))} chars)")
            except Exception as e:
                function_response = {"error": str(e)}
                if verbose:
                    print(f"   ✗ Error: {str(e)}")
            
            # Add function response to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "name": function_name,
                "content": json.dumps(function_response)
            })
            
            if verbose:
                print()
    
    return "Maximum iterations reached. Could not complete the query."


In [46]:
# ============================================================================
# EXAMPLE USAGE
# ============================================================================

# Example questions to test the agent
example_questions = [
    "What is the compatibility information for material 885600034?",
    "Show me all materials in compatibility group D with their NEW values",
    "What's the total inventory across all storage locations?",
    "Which storage location has the highest amount of explosives?",
    "Tell me about material 885600034 - its compatibility, inventory, and storage",
    "What would be the aggregated NEW if I store materials 885600034 and 885600011 together?",
    "Calculate the QTD values for storage location 00AJ"
]

print("\nAvailable Example Questions:")
for i, q in enumerate(example_questions, 1):
    print(f"{i}. {q}")


Available Example Questions:
1. What is the compatibility information for material 885600034?
2. Show me all materials in compatibility group D with their NEW values
3. What's the total inventory across all storage locations?
4. Which storage location has the highest amount of explosives?
5. Tell me about material 885600034 - its compatibility, inventory, and storage
6. What would be the aggregated NEW if I store materials 885600034 and 885600011 together?
7. Calculate the QTD values for storage location 00AJ


In [47]:
compatibility_test_answer = run_explosives_agent("What would be the aggregated NEW if I store materials 885600034 and 885600011 together?")

💥 EXPLOSIVES INTELLIGENCE AGENT
🔍 Question: What would be the aggregated NEW if I store materials 885600034 and 885600011 together?

🔧 Tool Call: calculate_materials_aggregation
   Parameters: {'material_numbers': [885600034, 885600011]}
   ✓ Response received (690 chars)

🔧 Tool Call: query_database_with_sql
   Parameters: {'user_query': 'Retrieve master data for material 885600011: description, hazard division, compatibility group, and NEW per unit (kg).'}
   ✓ Response received (334 chars)


✅ Final Answer:
Short answer: 0.0218 kg NEW — but that only includes material 885600034. Material 885600011 could not be found in the master data, so it was excluded from the calculation.

Details:
- Aggregation run result:
  - Included: 885600034 (POINT DETONATING FUZE), HD 1.2, NEW per unit: 0.0218 kg
  - Missing/not found: 885600011
  - IATG-aggregated HD (with only the included item): 1.2.1
  - Aggregated NEW (sum of included items): 0.0218 kg

What I need from you to complete this:
- Please

In [22]:
compatibility_test_answer = run_explosives_agent("I would like to store materials CHG, PROP 155MM M19 and RCKT MTR, DUAL THRUST MK104-3 ALUMINUM CNTR (37) together. Good idea?",
                                                 verbose = False) # Verbose can be turned off
print(compatibility_test_answer)

Short answer: No. These two materials should not be stored together.

Details from the explosives database:
- CHG, PROP 155MM M19
  - Material number: 880009632
  - Hazard division: 1.2
  - Compatibility group: K (articles containing both an explosive substance and a toxic component)
- RCKT MTR, DUAL THRUST MK104-3 ALUMINUM CNTR (37)
  - Material number: 887120843
  - Hazard division: 1.2.1
  - Compatibility group: C (propellant explosive substance)

Compatibility rule:
- Mixing CG K with CG C is not permitted. In the compatibility matrix, C-to-K is code 0, meaning “Not permitted.”
- CG K generally may only be stored with other CG K items (code X for K-to-K). There are no “segregated in same building” exceptions for K with other groups.

What this means operationally:
- Do not place these materials in the same magazine, room, or building.
- Store CHG, PROP 155MM M19 (CG K, HD 1.2) in a separate PES/magazine dedicated to CG K.
- Store the MK104-3 rocket motors (CG C, HD 1.2.1) with comp

### WebSocket Streaming Support

In [34]:
# ============================================================================
# API CLIENT EXAMPLES - SSE Streaming
# ============================================================================

import requests
import json

def test_sse_streaming(question, model="gpt-5"):
    """
    Test the SSE streaming endpoint with a question.
    Shows real-time agent thinking process using Server-Sent Events.
    """
    url = "https://explosives-api.cfapps.ap10.hana.ondemand.com/agent/stream"
    
    print(f"Connecting to {url}...")
    print(f"Question: {question}\n")
    
    try:
        # Make streaming POST request
        # Note: verify=False disables SSL verification for Cloud Foundry apps with underscore in hostname
        response = requests.post(
            url,
            params={"question": question, "model": model},
            stream=True,
            timeout=120
        )
        
        response.raise_for_status()
        
        # Process SSE stream
        for line in response.iter_lines():
            if line:
                line = line.decode('utf-8')
                
                # SSE format: "data: {json}"
                if line.startswith('data: '):
                    data = json.loads(line[6:])  # Remove 'data: ' prefix
                    event_type = data.get('type')
                    
                    if event_type == 'status':
                        print(f"{data['message']}")
                        
                    elif event_type == 'iteration':
                        print(f"\nIteration {data['iteration']}: {data['message']}")
                        
                    elif event_type == 'tool_call_start':
                        print(f"   Calling tool: {data['tool_name']}")
                        print(f"   Args: {data.get('parameters', {})}")
                        
                    elif event_type == 'tool_call_result':
                        result_preview = data.get('result_preview', '')
                        print(f"Tool result ({data.get('result_size', 0)} bytes): {result_preview}")
                        
                    elif event_type == 'final_answer':
                        print(f"\n{'='*80}")
                        print(f"FINAL ANSWER:")
                        print(f"{'='*80}")
                        print(data['content'])
                        print(f"{'='*80}\n")
                        break
                        
                    elif event_type == 'error':
                        print(f"Error: {data.get('message', str(data))}")
                        break
                    
                    elif event_type == 'tool_call_error':
                        print(f"Tool error in {data.get('tool_name')}: {data.get('error')}")
                    
    except requests.exceptions.RequestException as e:
        print(f"Connection error: {str(e)}")
        print("API URL: https://explosives-api.cfapps.ap10.hana.ondemand.com/")
    except Exception as e:
        print(f"Error: {str(e)}")


# Example questions to test
example_questions = [
    "What is the compatibility information for material 885600034?",
    "Can I store material 885600034 with material 885600011?",
    "Show me all materials in compatibility group D",
    "What is the total inventory across all storage locations?",
    "Which storage location has the highest amount of explosives?",
]

print("Available test questions:")
for i, q in enumerate(example_questions, 1):
    print(f"{i}. {q}")

# Run a test query
print("\n" + "="*80)
print("TESTING SSE STREAMING API")
print("="*80 + "\n")

# Test the SSE streaming
test_sse_streaming(example_questions[0])

Available test questions:
1. What is the compatibility information for material 885600034?
2. Can I store material 885600034 with material 885600011?
3. Show me all materials in compatibility group D
4. What is the total inventory across all storage locations?
5. Which storage location has the highest amount of explosives?

TESTING SSE STREAMING API

Connecting to https://explosives-api.cfapps.ap10.hana.ondemand.com/agent/stream...
Question: What is the compatibility information for material 885600034?

Starting explosives intelligence agent...

Iteration 1: Analyzing your question...
   Calling tool: get_compatibility_info
   Args: {'material_number': 885600034}
Tool result (1096 bytes): {'material_number': 885600034, 'material_description': 'POINT DETONATING FUZE', 'compatibility_group': 'B', 'compatibility_group_description': 'Articles containing a primary explosive substance', 'hazard_division': '1.2', 'compatibility_with_other_groups': {'A': '0', 'B': 'X', 'C': '1', 'D': '1', 'E..